In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import open3d as o3d

import configs as cfg
import os

In [5]:
mesh_csv = pd.read_csv( os.path.join(cfg.working_directory, "outputs", "mesh_area_volume.csv"))

mesh_csv.head()

,label,group_id,bucket_id,pin_id,area (cm2),volume (cm3)
0,R1-1,1,1,1,92.688152,79.719868
1,R1-2,1,1,2,188.680629,216.664250
2,R1-3,1,1,3,139.385540,142.948028
3,R1-4,1,1,4,99.121849,89.372959
4,R1-5,1,1,5,54.382122,36.550285


In [11]:
results = []

for index, row in mesh_csv.iterrows():
    db_mesh_folder = r"/home/crest/w/hwang_Pro/datasets/HuggingFace/3DPotatoTwin.unzip/2_sfm/3_mesh/"
    mesh_path = os.path.join(db_mesh_folder, row['label'], row['label'] + ".obj")

    mesh = o3d.io.read_triangle_mesh(mesh_path)
    bbox = mesh.get_axis_aligned_bounding_box()
    bbox_extent = bbox.get_extent()
    
    # 排序三个轴长
    sorted_extent = np.sort(bbox_extent)
    aspect_ratio = sorted_extent[2] / sorted_extent[0]  # 最长/最短轴比

    # 2. 获取表面积和体积
    surface_area = row['area (cm2)']
    volume = row['volume (cm3)']

    # 3. 计算球形度指数
    sphere_surface = (36 * np.pi * volume**2) ** (1/3)  # 等效球体表面积
    sphericity = sphere_surface / surface_area

    # 4. 计算体积表面积比
    volume_surface_ratio = volume / surface_area

    # 计算凸包
    convex_hull = mesh.compute_convex_hull()[0]  # 返回(TriangleMesh, vertex_indices)
    
    # 计算凸包体积
    convex_volume = convex_hull.get_volume() * 1000 * 1000
    convexity = volume / convex_volume

    # 存储结果
    results.append({
        'label': row['label'],
        'group_id': row['group_id'],
        'bucket_id': row['bucket_id'],
        'pin_id': row['pin_id'],
        'major axis length (cm)': bbox_extent[0] * 100,
        'mid axis length (cm)': bbox_extent[1] * 100,
        'minor axis length (cm)': bbox_extent[2] * 100,
        'area (cm2)': surface_area,
        'volume (cm3)': volume,
        'aspect ratio': aspect_ratio,
        'volume/surface Ratio': volume_surface_ratio,
        'sphericity': sphericity,
        'convexity': convexity
    })

    break


# results_pd = pd.DataFrame(results)

In [12]:
results

[{'label': 'R1-1',
  'group_id': 1,
  'bucket_id': 1,
  'pin_id': 1,
  'major axis length (cm)': 4.90300003439188,
  'mid axis length (cm)': 4.845299944281578,
  'minor axis length (cm)': 6.909200549125671,
  'area (cm2)': 92.6881523806474,
  'volume (cm3)': 79.7198675285296,
  'aspect ratio': 1.4259593066637513,
  'volume/surface Ratio': 0.8600869202909532,
  'sphericity': 0.9664312061616495,
  'convexity': 0.9909303591402766}]

In [13]:
from tqdm import tqdm

In [17]:
results = []

for index, row in tqdm( mesh_csv.iterrows(), total=len(mesh_csv), desc="Processing meshes" ):
    db_mesh_folder = r"/home/crest/w/hwang_Pro/datasets/HuggingFace/3DPotatoTwin.unzip/2_sfm/3_mesh/"
    mesh_path = os.path.join(db_mesh_folder, row['label'], row['label'] + ".obj")

    if not os.path.exists(mesh_path):
        print(f"{row['label']} mesh file not exists")
        continue

    mesh = o3d.io.read_triangle_mesh(mesh_path)
    bbox = mesh.get_axis_aligned_bounding_box()
    bbox_extent = bbox.get_extent()
    
    # 排序三个轴长
    sorted_extent = np.sort(bbox_extent)
    aspect_ratio = sorted_extent[2] / sorted_extent[0]  # 最长/最短轴比

    # 2. 获取表面积和体积
    surface_area = row['area (cm2)']
    volume = row['volume (cm3)']

    # 3. 计算球形度指数
    sphere_surface = (36 * np.pi * volume**2) ** (1/3)  # 等效球体表面积
    sphericity = sphere_surface / surface_area

    # 4. 计算体积表面积比
    volume_surface_ratio = volume / surface_area

    # 计算凸包
    convex_hull = mesh.compute_convex_hull()[0]  # 返回(TriangleMesh, vertex_indices)
    
    # 计算凸包体积
    convex_volume = convex_hull.get_volume() * 1000 * 1000
    convexity = volume / convex_volume

    # 存储结果
    results.append({
        'label': row['label'],
        'group_id': row['group_id'],
        'bucket_id': row['bucket_id'],
        'pin_id': row['pin_id'],
        'minor axis length (cm)': bbox_extent[0] * 100,
        'middle axis length (cm)': bbox_extent[1] * 100,
        'major axis length (cm)': bbox_extent[2] * 100,
        'area (cm2)': surface_area,
        'volume (cm3)': volume,
        'aspect ratio': aspect_ratio,
        'volume/surface Ratio': volume_surface_ratio,
        'sphericity': sphericity,
        'convexity': convexity
    })

results_pd = pd.DataFrame(results)

Processing meshes:  52%|█████▏    | 179/341 [00:55<00:43,  3.73it/s]

3R2-7 mesh file not exists


Processing meshes:  55%|█████▌    | 189/341 [00:58<00:38,  4.00it/s]

3R3-7 mesh file not exists


Processing meshes: 100%|██████████| 341/341 [01:47<00:00,  3.18it/s]


In [18]:
results_pd

,label,group_id,bucket_id,pin_id,minor axis length (cm),middle axis length (cm),major axis length (cm),area (cm2),volume (cm3),aspect ratio,volume/surface Ratio,sphericity,convexity
0,R1-1,1,1,1,4.9030,4.8453,6.909201,92.688152,79.719868,1.425959,0.860087,0.966431,0.990930
1,R1-2,1,1,2,6.9880,5.7693,10.586900,188.680629,216.664250,1.835041,1.148312,0.924588,0.980569
2,R1-3,1,1,3,6.0079,6.0573,7.861100,139.385540,142.948028,1.308461,1.025559,0.948532,0.966446
3,R1-4,1,1,4,5.3464,5.1903,6.378900,99.121849,89.372959,1.229004,0.901647,0.975256,0.992565
4,R1-5,1,1,5,3.9682,3.9633,4.925101,54.382122,36.550285,1.242677,0.672101,0.979389,0.986339
...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,5R3-6,5,3,6,4.8903,4.3957,7.291899,95.020869,79.426222,1.658871,0.835882,0.940389,0.984048
335,5R3-7,5,3,7,3.7750,3.9932,6.642900,71.473405,51.327648,1.759708,0.718136,0.934488,0.991057
336,5R3-8,5,3,8,4.7262,4.8177,8.811700,112.212137,95.468087,1.864437,0.850782,0.900222,0.960685
337,5R3-9,5,3,9,5.8528,5.9733,9.911700,162.180051,175.928857,1.693497,1.084775,0.936220,0.983940


In [19]:
results_pd.to_csv(os.path.join(cfg.working_directory, "outputs", "mesh_traits.csv"), index=None)